# Lab 05: SVD, Pseudoinverse, Trace & Determinant

**Reference:** Goodfellow et al. *Deep Learning*, Chapter 2, Sections 2.8–2.11

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
print('Setup complete.')

## Part 1: Singular Value Decomposition (Section 2.8)

Every real matrix $A \in \mathbb{R}^{m \times n}$ can be factored as:
$$A = U \Sigma V^T$$
where $U \in \mathbb{R}^{m \times m}$ and $V \in \mathbb{R}^{n \times n}$ are orthogonal matrices, and $\Sigma \in \mathbb{R}^{m \times n}$ is diagonal with non-negative entries (singular values) $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$.

In [ ]:
# Demo: Geometric meaning of SVD
# A = U * Sigma * V^T acts on the unit circle in 3 steps:
#   Step 1: V^T rotates/reflects
#   Step 2: Sigma scales along axes
#   Step 3: U rotates/reflects

theta = np.linspace(0, 2 * np.pi, 300)
unit_circle = np.array([np.cos(theta), np.sin(theta)])  # shape (2, 300)

# A random 2x2 matrix
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])

U, s, Vt = np.linalg.svd(A)
Sigma = np.diag(s)

# Apply each step
step1 = Vt @ unit_circle          # after V^T
step2 = Sigma @ step1             # after Sigma
step3 = U @ step2                 # after U  (= A @ unit_circle)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, pts, title in zip(
    axes,
    [unit_circle, step1, step2, step3],
    ['Original unit circle', r'After $V^T$ (rotation)', r'After $\Sigma$ (scaling)', r'After $U\Sigma V^T = A$ (rotation)']):
    ax.plot(pts[0], pts[1], 'b-', linewidth=2)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)

plt.suptitle('SVD Geometric Interpretation: A = U Σ Vᵀ', fontsize=12)
plt.tight_layout()
plt.show()

print(f'A =\n{A}')
print(f'Singular values: {s}')
print(f'U =\n{U}')
print(f'V =\n{Vt.T}')

### Exercise 1 (Basic): SVD Verification

Compute the SVD of a rectangular matrix, reconstruct $A$ from $U$, $\Sigma$, $V^T$, and verify that $U$ and $V$ are orthogonal (i.e., $U^T U = I$ and $V^T V = I$).

In [ ]:
# Exercise 1: SVD verification
A = np.random.randn(4, 3)

# 1. Compute the full SVD of A using np.linalg.svd with full_matrices=True
U, s, Vt = np.linalg.svd(A, full_matrices=True)

# 2. Build the full Sigma matrix (shape m x n) from the singular values vector s
m_A, n_A = A.shape
Sigma_full = np.zeros((m_A, n_A))
np.fill_diagonal(Sigma_full, s)

# 3. Reconstruct A_reconstructed = U @ Sigma_full @ Vt
A_reconstructed = U @ Sigma_full @ Vt

# 4. Compute U^T U - I and V^T V - I
V = Vt.T
U_ortho_error = U.T @ U - np.eye(U.shape[0])
V_ortho_error = V.T @ V - np.eye(V.shape[0])
recon_error = np.max(np.abs(A_reconstructed - A))

print(f'A shape: {A.shape}')
print(f'U shape: {U.shape}, s shape: {s.shape}, Vt shape: {Vt.shape}')
print(f'Reconstruction error (max |A - U Sigma Vt|): {recon_error:.2e}')
print(f'U orthogonality error (max |U^T U - I|): {np.max(np.abs(U_ortho_error)):.2e}')
print(f'V orthogonality error (max |V^T V - I|): {np.max(np.abs(V_ortho_error)):.2e}')

In [ ]:
# Verification cell for Exercise 1
assert U.shape == (4, 4), f'U should be 4x4, got {U.shape}'
assert s.shape == (3,), f's should have 3 singular values, got {s.shape}'
assert Vt.shape == (3, 3), f'Vt should be 3x3, got {Vt.shape}'
assert np.allclose(A_reconstructed, A, atol=1e-10), 'Reconstruction A = U Sigma Vt failed'
assert np.allclose(U_ortho_error, np.zeros((4,4)), atol=1e-10), 'U is not orthogonal'
assert np.allclose(V_ortho_error, np.zeros((3,3)), atol=1e-10), 'V is not orthogonal'
print('Exercise 1 PASSED')
print(f'  Reconstruction error: {recon_error:.2e}')
print(f'  U orthogonality error (max |U^T U - I|): {np.max(np.abs(U_ortho_error)):.2e}')
print(f'  V orthogonality error (max |V^T V - I|): {np.max(np.abs(V_ortho_error)):.2e}')

### Exercise 2 (Basic): SVD–Eigendecomposition Relationship

The singular values of $A$ are the square roots of the eigenvalues of $A^T A$, and the right singular vectors $V$ are the eigenvectors of $A^T A$.

In [ ]:
# Exercise 2: SVD-eigendecomposition relationship
A = np.random.randn(5, 3)

# 1. Compute SVD of A (economy/thin SVD: full_matrices=False)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# 2. Compute eigendecomposition of A^T A using np.linalg.eigh (symmetric matrix)
#    Note: eigh returns eigenvalues in ascending order; sort descending
AtA = A.T @ A
eigvals, eigvecs = np.linalg.eigh(AtA)

# Sort descending
idx = np.argsort(eigvals)[::-1]
eigvals_sorted = eigvals[idx]
eigvecs_sorted = eigvecs[:, idx]

# 3. Verify: singular values s == sqrt(eigenvalues of A^T A)
sv_from_eig = np.sqrt(np.maximum(eigvals_sorted, 0.0))

# Compare singular values
singular_val_match = s - sv_from_eig

# 4. Compare V columns (allow sign flips: compare abs of dot products with 1)
V = Vt.T
vec_alignment = np.array([
    abs(np.dot(V[:, j], eigvecs_sorted[:, j]))
    for j in range(V.shape[1])
])

In [ ]:
# Verification cell for Exercise 2
assert np.allclose(singular_val_match, np.zeros_like(s), atol=1e-10), \
    f'Singular values do not match sqrt(eigenvalues of A^T A). Diff: {singular_val_match}'
assert np.allclose(vec_alignment, np.ones(V.shape[1]), atol=1e-10), \
    f'V columns do not match eigenvectors of A^T A. Alignments: {vec_alignment}'
print('Exercise 2 PASSED')
print(f'  Singular values from SVD:         {s}')
print(f'  Singular values from sqrt(eig):   {sv_from_eig}')
print(f'  Max difference:                   {np.max(np.abs(singular_val_match)):.2e}')
print(f'  V column alignments (should be 1): {vec_alignment}')

### Exercise 3 (Intermediate): Low-Rank Approximation (Eckart–Young Theorem)

The best rank-$k$ approximation to $A$ in Frobenius norm is the truncated SVD:
$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$
The approximation error is:
$$\|A - A_k\|_F = \sqrt{\sum_{i=k+1}^{r} \sigma_i^2}$$

In [ ]:
# Exercise 3: Low-rank approximation and Eckart-Young theorem
A = np.random.randn(8, 6)

def truncated_svd(U, s, Vt, k):
    """Return the best rank-k approximation of A = U diag(s) Vt."""
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

U, s, Vt = np.linalg.svd(A, full_matrices=False)

ks = list(range(1, min(A.shape) + 1))
actual_errors = []
theoretical_errors = []

for k in ks:
    A_k = truncated_svd(U, s, Vt, k)
    actual_err = np.linalg.norm(A - A_k, 'fro')
    theoretical_err = np.sqrt(np.sum(s[k:] ** 2))
    actual_errors.append(actual_err)
    theoretical_errors.append(theoretical_err)

plt.figure(figsize=(8, 4))
plt.plot(ks, actual_errors, 'bo-', label='Actual ||A - A_k||_F')
plt.plot(ks, theoretical_errors, 'r--', marker='x', label=r'Theoretical $\sqrt{\sum_{i>k}\sigma_i^2}$')
plt.xlabel('Rank k')
plt.ylabel('Frobenius Error')
plt.title('Eckart–Young Theorem: Low-Rank Approximation Error')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verification cell for Exercise 3
for i, k in enumerate(ks):
    assert np.isclose(actual_errors[i], theoretical_errors[i], atol=1e-10), \
        f'Eckart-Young failed at k={k}: actual={actual_errors[i]:.6f}, theoretical={theoretical_errors[i]:.6f}'
print('Exercise 3 PASSED: Eckart-Young theorem verified for all k')
for k, ae, te in zip(ks, actual_errors, theoretical_errors):
    print(f'  k={k}: actual={ae:.6f}, theoretical={te:.6f}, diff={abs(ae-te):.2e}')

### Exercise 4 (Intermediate): Image Compression with SVD

SVD-based compression stores rank-$k$ approximation using $k(m + n + 1)$ numbers instead of $mn$, giving compression ratio $\frac{mn}{k(m+n+1)}$.

In [ ]:
# Exercise 4: Image compression with SVD
# Create a synthetic grayscale image with structure
m, n = 64, 64
xx, yy = np.meshgrid(np.linspace(0, 4*np.pi, n), np.linspace(0, 4*np.pi, m))
image = (np.sin(xx) * np.cos(yy) + np.sin(2*xx) * np.cos(3*yy) +
         0.3 * np.random.randn(m, n))
image = (image - image.min()) / (image.max() - image.min())  # normalize to [0,1]

# 1. Compute the full SVD of the image
U, s, Vt = np.linalg.svd(image, full_matrices=False)

ranks = [1, 5, 10, 20]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original\n(64x64)', fontsize=9)
axes[0].axis('off')

for ax, k in zip(axes[1:], ranks):
    # 2a. Compute the rank-k approximation
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    # 2b. Clip values to [0, 1]
    approx_clipped = np.clip(approx, 0, 1)
    # 2c. Compute compression ratio
    compression_ratio = (m * n) / (k * (m + n + 1))
    # 2d. Compute PSNR
    mse = np.mean((image - approx_clipped) ** 2)
    psnr = 20 * np.log10(1.0 / np.sqrt(mse)) if mse > 0 else float('inf')

    ax.imshow(approx_clipped, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Rank {k}\nCR={compression_ratio:.1f}x\nPSNR={psnr:.1f}dB', fontsize=9)
    ax.axis('off')

plt.suptitle('SVD Image Compression', fontsize=12)
plt.tight_layout()
plt.show()

# Print singular value decay
print(f'Top 20 singular values: {s[:20].round(2)}')
print(f'Energy captured by top 20 singular values: {(s[:20]**2).sum() / (s**2).sum() * 100:.1f}%')

## Part 2: Moore–Penrose Pseudoinverse (Section 2.9)

For a matrix $A$ with SVD $A = U \Sigma V^T$, the Moore–Penrose pseudoinverse is:
$$A^+ = V \Sigma^+ U^T$$
where $\Sigma^+$ replaces each non-zero $\sigma_i$ with $1/\sigma_i$ (and leaves zeros as zero).

### Exercise 5 (Basic): Implement the Pseudoinverse via SVD

Implement $A^+ = V \Sigma^+ U^T$ and compare to `np.linalg.pinv`.

In [ ]:
# Exercise 5: Pseudoinverse via SVD

def pseudoinverse(A, tol=None):
    """Compute Moore-Penrose pseudoinverse via SVD."""
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    m_loc, n_loc = A.shape
    if tol is None:
        eps = np.finfo(float).eps
        tol = max(m_loc, n_loc) * eps * s[0]
    s_inv = np.where(s > tol, 1.0 / s, 0.0)
    return Vt.T @ np.diag(s_inv) @ U.T

# Test on tall (overdetermined) matrix
A_tall = np.random.randn(6, 3)
A_pinv_custom = pseudoinverse(A_tall)
A_pinv_numpy = np.linalg.pinv(A_tall)

tall_error = np.max(np.abs(A_pinv_custom - A_pinv_numpy))

# Test on wide (underdetermined) matrix
A_wide = np.random.randn(3, 6)
A_wide_pinv_custom = pseudoinverse(A_wide)
A_wide_pinv_numpy = np.linalg.pinv(A_wide)

wide_error = np.max(np.abs(A_wide_pinv_custom - A_wide_pinv_numpy))

# Test on rank-deficient matrix
A_rank_def = np.random.randn(4, 2) @ np.random.randn(2, 4)  # rank 2 matrix (4x4)
A_rd_pinv_custom = pseudoinverse(A_rank_def)
A_rd_pinv_numpy = np.linalg.pinv(A_rank_def)

rank_def_error = np.max(np.abs(A_rd_pinv_custom - A_rd_pinv_numpy))

In [ ]:
# Verification cell for Exercise 5
assert np.allclose(A_pinv_custom, A_pinv_numpy, atol=1e-10), \
    f'Pseudoinverse of tall matrix wrong. Max error: {tall_error:.2e}'
assert np.allclose(A_wide_pinv_custom, A_wide_pinv_numpy, atol=1e-10), \
    f'Pseudoinverse of wide matrix wrong. Max error: {wide_error:.2e}'
assert np.allclose(A_rd_pinv_custom, A_rd_pinv_numpy, atol=1e-10), \
    f'Pseudoinverse of rank-deficient matrix wrong. Max error: {rank_def_error:.2e}'
print('Exercise 5 PASSED')
print(f'  Tall matrix max error:          {tall_error:.2e}')
print(f'  Wide matrix max error:          {wide_error:.2e}')
print(f'  Rank-deficient matrix max error:{rank_def_error:.2e}')

### Exercise 6 (Intermediate): Least Squares via Pseudoinverse

For an overdetermined system $Ax = b$ (more equations than unknowns), the pseudoinverse gives the least-squares solution: $x^* = A^+ b$, which minimizes $\|Ax - b\|_2^2$.

In [ ]:
# Exercise 6: Least squares via pseudoinverse
# Overdetermined system: 10 equations, 3 unknowns
m, n = 10, 3
A = np.random.randn(m, n)
x_true = np.array([1.0, -2.0, 3.0])
b = A @ x_true + 0.1 * np.random.randn(m)  # add noise -> no exact solution

# 1. Solve via pseudoinverse: x_pinv = A^+ @ b
x_pinv = pseudoinverse(A) @ b

# 2. Solve via np.linalg.lstsq
x_lstsq, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)

# 3. Solve via normal equations: x_normal = (A^T A)^{-1} A^T b
x_normal = np.linalg.solve(A.T @ A, A.T @ b)

# 4. Compute residuals
residual_pinv = np.linalg.norm(A @ x_pinv - b)
residual_lstsq = np.linalg.norm(A @ x_lstsq - b)
residual_normal = np.linalg.norm(A @ x_normal - b)

In [ ]:
# Verification cell for Exercise 6
assert np.allclose(x_pinv, x_lstsq, atol=1e-8), \
    f'Pseudoinverse and lstsq solutions differ. Max diff: {np.max(np.abs(x_pinv - x_lstsq)):.2e}'
assert np.allclose(x_pinv, x_normal, atol=1e-8), \
    f'Pseudoinverse and normal equations solutions differ. Max diff: {np.max(np.abs(x_pinv - x_normal)):.2e}'
assert np.isclose(residual_pinv, residual_lstsq, atol=1e-8), \
    f'Residuals differ: pinv={residual_pinv:.6f}, lstsq={residual_lstsq:.6f}'
print('Exercise 6 PASSED: All three methods give the same least-squares solution')
print(f'  x_pinv:   {x_pinv}')
print(f'  x_lstsq:  {x_lstsq}')
print(f'  x_normal: {x_normal}')
print(f'  Residual ||Ax - b||_2 = {residual_pinv:.6f}')

### Exercise 7 (Intermediate): Minimum Norm Solution

For an underdetermined system $Ax = b$ (more unknowns than equations, infinitely many solutions), the pseudoinverse gives the minimum $L_2$-norm solution: $x^* = A^+ b$, i.e., $\|x^*\|_2 \leq \|x\|_2$ for all other solutions.

In [ ]:
# Exercise 7: Minimum norm solution for underdetermined systems
# Underdetermined: 3 equations, 6 unknowns -> infinitely many solutions
m, n = 3, 6
A = np.random.randn(m, n)
x_particular = np.random.randn(n)
b = A @ x_particular  # exact solution exists

# 1. Compute x_minnorm = A^+ b (minimum norm solution)
x_minnorm = pseudoinverse(A) @ b

# 2. Verify it satisfies Ax = b exactly
residual_check = A @ x_minnorm - b

# 3. Generate null space vectors using (I - A^+ A)
A_pinv = np.linalg.pinv(A)
P_null = np.eye(n) - A_pinv @ A  # projection onto null space of A

N = 1000
norms_random = []
for _ in range(N):
    v = np.random.randn(n)
    x_rand = x_minnorm + P_null @ v
    norms_random.append(np.linalg.norm(x_rand))

norms_random = np.array(norms_random)
norm_minnorm = np.linalg.norm(x_minnorm)

plt.figure(figsize=(8, 4))
plt.hist(norms_random, bins=40, alpha=0.7, label='Random solutions')
plt.axvline(norm_minnorm, color='r', linewidth=2, label=f'||x_minnorm|| = {norm_minnorm:.3f}')
plt.xlabel('||x||_2')
plt.ylabel('Count')
plt.title('Minimum Norm Solution: Pseudoinverse gives smallest ||x||')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verification cell for Exercise 7
assert np.allclose(residual_check, np.zeros(m), atol=1e-10), \
    f'Minimum norm solution does not satisfy Ax = b. Residual: {residual_check}'
assert np.all(norms_random >= norm_minnorm - 1e-10), \
    f'Some random solution has smaller norm than pseudoinverse solution! Min random norm: {norms_random.min():.6f}, minnorm: {norm_minnorm:.6f}'
print('Exercise 7 PASSED')
print(f'  ||x_minnorm||_2 = {norm_minnorm:.6f}')
print(f'  Min ||x_random||_2 = {norms_random.min():.6f}')
print(f'  Mean ||x_random||_2 = {norms_random.mean():.6f}')
print(f'  Ax = b residual (should be ~0): {np.max(np.abs(residual_check)):.2e}')

## Part 3: Trace (Section 2.10)

The trace of a square matrix $A \in \mathbb{R}^{n \times n}$ is the sum of its diagonal elements:
$$\text{tr}(A) = \sum_{i=1}^{n} A_{ii}$$

Key properties:
- $\text{tr}(A) = \sum_i \lambda_i$ (sum of eigenvalues)
- $\text{tr}(AB) = \text{tr}(BA)$ (cyclic property, even for non-square $A$, $B$)
- $\text{tr}(A^T B) = \sum_{i,j} A_{ij} B_{ij}$ (Frobenius inner product)

### Exercise 8 (Basic): Trace Properties

Implement trace from scratch and verify three key properties.

In [ ]:
# Exercise 8: Trace properties

def trace(A):
    """Compute trace of square matrix A without np.trace."""
    return sum(A[i, i] for i in range(min(A.shape)))

# (a) tr(A) = sum of eigenvalues
A = np.random.randn(5, 5)
trace_A = trace(A)
eigenvalues = np.linalg.eigvals(A)
trace_from_eig = np.real(np.sum(eigenvalues))  # sum of eigenvalues (real part)

# (b) tr(AB) = tr(BA) for non-square A, B
A_rect = np.random.randn(4, 6)
B_rect = np.random.randn(6, 4)
trace_AB = trace(A_rect @ B_rect)  # 4x4 matrix
trace_BA = trace(B_rect @ A_rect)  # 6x6 matrix

# (c) tr(A^T B) = Frobenius inner product sum_{i,j} A_{ij} B_{ij}
A2 = np.random.randn(4, 5)
B2 = np.random.randn(4, 5)
trace_AtB = trace(A2.T @ B2)       # trace of A^T @ B
frobenius_inner = np.sum(A2 * B2)  # element-wise sum A2 * B2

In [ ]:
# Verification cell for Exercise 8
assert np.isclose(trace_A, np.trace(A), atol=1e-10), \
    f'trace() implementation wrong. Got {trace_A}, expected {np.trace(A)}'
assert np.isclose(trace_A, trace_from_eig, atol=1e-10), \
    f'tr(A) != sum(eigenvalues). Got {trace_A} vs {trace_from_eig}'
assert np.isclose(trace_AB, trace_BA, atol=1e-10), \
    f'tr(AB) != tr(BA). Got {trace_AB} vs {trace_BA}'
assert np.isclose(trace_AtB, frobenius_inner, atol=1e-10), \
    f'tr(A^T B) != Frobenius inner product. Got {trace_AtB} vs {frobenius_inner}'
print('Exercise 8 PASSED: All trace properties verified')
print(f'  (a) tr(A) = {trace_A:.6f}, sum(eigenvalues) = {trace_from_eig:.6f}')
print(f'  (b) tr(AB) = {trace_AB:.6f}, tr(BA) = {trace_BA:.6f}')
print(f'  (c) tr(A^T B) = {trace_AtB:.6f}, Frobenius inner = {frobenius_inner:.6f}')

### Exercise 9 (Intermediate): Trace Trick in ML — Gradient Verification

A useful identity in ML (e.g., Gaussian log-likelihood): $\frac{d}{dA} \text{tr}(AB) = B^T$.

Verify this numerically using finite differences.

In [ ]:
# Exercise 9: Trace trick - verify d/dA tr(AB) = B^T via finite differences
A = np.random.randn(4, 4)
B = np.random.randn(4, 4)

def f(A, B):
    """Compute tr(A @ B)."""
    return np.trace(A @ B)

# 1. Analytical gradient: d/dA tr(AB) = B^T
grad_analytical = B.T

# 2. Numerical gradient via central finite differences
eps = 1e-5
grad_numerical = np.zeros_like(A)

for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        A_plus = A.copy()
        A_plus[i, j] += eps
        A_minus = A.copy()
        A_minus[i, j] -= eps
        grad_numerical[i, j] = (f(A_plus, B) - f(A_minus, B)) / (2 * eps)

max_grad_error = np.max(np.abs(grad_analytical - grad_numerical))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
im1 = axes[0].imshow(grad_analytical, cmap='RdBu_r')
axes[0].set_title('Analytical: B^T')
plt.colorbar(im1, ax=axes[0])
im2 = axes[1].imshow(grad_numerical, cmap='RdBu_r')
axes[1].set_title('Numerical (finite diff)')
plt.colorbar(im2, ax=axes[1])
im3 = axes[2].imshow(grad_analytical - grad_numerical, cmap='RdBu_r')
axes[2].set_title(f'Error (max={max_grad_error:.2e})')
plt.colorbar(im3, ax=axes[2])
plt.suptitle(r'Gradient of tr(AB) w.r.t. A', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Verification cell for Exercise 9
assert np.allclose(grad_analytical, grad_numerical, atol=1e-7), \
    f'd/dA tr(AB) = B^T verification failed. Max error: {max_grad_error:.2e}'
print('Exercise 9 PASSED: d/dA tr(AB) = B^T verified numerically')
print(f'  Max error between analytical and numerical gradient: {max_grad_error:.2e}')

## Part 4: Determinant (Section 2.11)

The determinant $\det(A)$ of a square matrix measures the signed volume scaling factor of the linear transformation $A$. Key properties:
- $\det(AB) = \det(A)\det(B)$
- $\det(A^{-1}) = 1/\det(A)$
- $\det(cA) = c^n \det(A)$ for $A \in \mathbb{R}^{n \times n}$
- $\det(A^T) = \det(A)$
- $\det(A) = \prod_i \lambda_i$ (product of eigenvalues)

### Exercise 10 (Basic): Determinant Properties

In [ ]:
# Exercise 10: Verify determinant properties
np.random.seed(7)
A = np.random.randn(4, 4)
B = np.random.randn(4, 4)
n = 4
c = 3.0

# Compute all determinants needed
det_A = np.linalg.det(A)
det_B = np.linalg.det(B)

# (a) det(AB) = det(A) * det(B)
det_AB = np.linalg.det(A @ B)
det_AB_product = det_A * det_B

# (b) det(A^{-1}) = 1 / det(A)
det_Ainv = np.linalg.det(np.linalg.inv(A))
det_A_recip = 1.0 / det_A

# (c) det(cA) = c^n * det(A)
det_cA = np.linalg.det(c * A)
det_cA_formula = (c ** n) * det_A

# (d) det(A^T) = det(A)
det_AT = np.linalg.det(A.T)

In [ ]:
# Verification cell for Exercise 10
assert np.isclose(det_AB, det_AB_product, rtol=1e-10), \
    f'det(AB) != det(A)*det(B): {det_AB:.6f} vs {det_AB_product:.6f}'
assert np.isclose(det_Ainv, det_A_recip, rtol=1e-10), \
    f'det(A^-1) != 1/det(A): {det_Ainv:.6f} vs {det_A_recip:.6f}'
assert np.isclose(det_cA, det_cA_formula, rtol=1e-10), \
    f'det(cA) != c^n det(A): {det_cA:.6f} vs {det_cA_formula:.6f}'
assert np.isclose(det_AT, det_A, rtol=1e-10), \
    f'det(A^T) != det(A): {det_AT:.6f} vs {det_A:.6f}'
print('Exercise 10 PASSED: All determinant properties verified')
print(f'  (a) det(AB)={det_AB:.4f}, det(A)det(B)={det_AB_product:.4f}')
print(f'  (b) det(A^-1)={det_Ainv:.4f}, 1/det(A)={det_A_recip:.4f}')
print(f'  (c) det(cA)={det_cA:.4f}, c^n det(A)={det_cA_formula:.4f}, c={c}, n={n}')
print(f'  (d) det(A^T)={det_AT:.4f}, det(A)={det_A:.4f}')

### Exercise 11 (Intermediate): Determinant via Cofactor Expansion

The cofactor expansion computes $\det(A) = \sum_j (-1)^{i+j} A_{ij} \det(M_{ij})$ where $M_{ij}$ is the minor obtained by deleting row $i$ and column $j$. This runs in $O(n!)$ time.

In [ ]:
# Exercise 11: Recursive cofactor expansion determinant

def det_cofactor(A):
    """Compute determinant via cofactor expansion (recursive, O(n!))."""
    A = np.array(A, dtype=float)
    n = A.shape[0]
    if n == 1:
        return A[0, 0]
    if n == 2:
        return A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    d = 0.0
    for j in range(n):
        minor = np.delete(np.delete(A, 0, axis=0), j, axis=1)
        d += ((-1) ** j) * A[0, j] * det_cofactor(minor)
    return d

# Verify correctness
for n in [2, 3, 4, 5]:
    M = np.random.randn(n, n)
    det_custom = det_cofactor(M)
    det_numpy = np.linalg.det(M)
    assert np.isclose(det_custom, det_numpy, rtol=1e-8), \
        f'det_cofactor wrong for n={n}: got {det_custom}, expected {det_numpy}'
    print(f'  n={n}: det_cofactor={det_custom:.6f}, np.linalg.det={det_numpy:.6f} ✓')

# Time cofactor expansion for n = 2..10 to show O(n!) growth
print('\nTiming cofactor expansion:')
import math
sizes = list(range(2, 11))
times_cofactor = []

for n in sizes:
    M = np.random.randn(n, n)
    reps = max(1, 1000 // (math.factorial(n) + 1))
    start = time.perf_counter()
    for _ in range(max(reps, 1)):
        det_cofactor(M)
    elapsed = (time.perf_counter() - start) / max(reps, 1)
    times_cofactor.append(elapsed)
    print(f'  n={n:2d}: {elapsed*1e6:.2f} μs')

plt.figure(figsize=(8, 4))
plt.semilogy(sizes, times_cofactor, 'bo-', label='Cofactor expansion')
plt.xlabel('Matrix size n')
plt.ylabel('Time (seconds, log scale)')
plt.title('Cofactor Expansion: O(n!) Runtime Growth')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verification cell for Exercise 11
for n in [2, 3, 4, 5, 6]:
    M = np.random.randn(n, n)
    assert np.isclose(det_cofactor(M), np.linalg.det(M), rtol=1e-8), \
        f'det_cofactor failed for n={n}'
print('Exercise 11 PASSED: Cofactor expansion correct for n=2..6')
print(f'  Time ratios (should grow roughly as (n+1)! / n! = n+1):')
for i in range(1, len(times_cofactor)):
    ratio = times_cofactor[i] / times_cofactor[i-1] if times_cofactor[i-1] > 0 else float('nan')
    print(f'    n={sizes[i-1]}->{sizes[i]}: ratio={ratio:.1f} (expected ~{sizes[i]})')

### Exercise 12 (Challenging): LU-Based Determinant

Using LU decomposition $PA = LU$:
$$\det(A) = \text{sign}(P) \cdot \prod_i U_{ii}$$
where $\text{sign}(P) = (-1)^{\text{number of row swaps}}$. This runs in $O(n^3)$.

In [ ]:
# Exercise 12: LU-based determinant
from scipy.linalg import lu

def det_lu(A):
    """Compute determinant via LU decomposition (O(n^3))."""
    P, L, U = lu(A)
    # det(L) = 1 (unit lower triangular)
    det_U = np.prod(np.diag(U))
    # det(P) = +1 or -1 (permutation matrix)
    det_P = np.linalg.det(P)
    return det_P * det_U

# Verify correctness on various sizes
print('Correctness check:')
for n in [3, 5, 8, 10]:
    M = np.random.randn(n, n)
    det_lu_val = det_lu(M)
    det_np_val = np.linalg.det(M)
    assert np.isclose(det_lu_val, det_np_val, rtol=1e-8), \
        f'det_lu wrong for n={n}: got {det_lu_val:.6f}, expected {det_np_val:.6f}'
    print(f'  n={n:2d}: det_lu={det_lu_val:.6f}, np.det={det_np_val:.6f} ✓')

# Speed comparison: LU vs cofactor
print('\nSpeed comparison (microseconds):')
for n in [4, 6, 8, 10]:
    M = np.random.randn(n, n)
    reps = 200

    start = time.perf_counter()
    for _ in range(reps):
        det_lu(M)
    t_lu = (time.perf_counter() - start) / reps * 1e6

    if n <= 8:
        start = time.perf_counter()
        for _ in range(reps):
            det_cofactor(M)
        t_cofactor = (time.perf_counter() - start) / reps * 1e6
        print(f'  n={n}: LU={t_lu:.1f}μs, Cofactor={t_cofactor:.1f}μs, Speedup={t_cofactor/t_lu:.1f}x')
    else:
        print(f'  n={n}: LU={t_lu:.1f}μs, Cofactor=too slow')

In [ ]:
# Verification cell for Exercise 12
for n in [3, 5, 8, 12, 15]:
    M = np.random.randn(n, n)
    assert np.isclose(det_lu(M), np.linalg.det(M), rtol=1e-8), \
        f'det_lu failed for n={n}'
print('Exercise 12 PASSED: LU-based determinant correct for n=3,5,8,12,15')

### Exercise 13 (Challenging): Volume Interpretation of the Determinant

The absolute value of the determinant gives the volume of the parallelepiped formed by the columns (or rows) of the matrix. In 2D: $|\det(A)|$ = area of parallelogram formed by columns of $A$ applied to the unit square.

In [ ]:
# Exercise 13: Volume interpretation of determinant

# ---- 2D: Unit square -> parallelogram ----
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Unit square corners (close the polygon)
unit_square = np.array([[0,1,1,0,0],
                         [0,0,1,1,0]], dtype=float)

matrices_2d = [
    ('Identity', np.eye(2)),
    ('Scaling (2,3)', np.array([[2.,0],[0,3.]])),
    ('Rotation 45°', np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                [np.sin(np.pi/4),  np.cos(np.pi/4)]])),
    ('Shear', np.array([[1.,2.],[0.,1.]])),
]

for col, (name, A2) in enumerate(matrices_2d):
    # Transform the unit square
    transformed = A2 @ unit_square
    det_val = np.linalg.det(A2)
    area = abs(det_val)  # area = |det(A)|

    ax_orig = axes[0][col]
    ax_trans = axes[1][col]

    ax_orig.fill(unit_square[0], unit_square[1], alpha=0.3, color='blue')
    ax_orig.plot(unit_square[0], unit_square[1], 'b-')
    ax_orig.set_xlim(-0.5, 3.5)
    ax_orig.set_ylim(-0.5, 3.5)
    ax_orig.set_aspect('equal')
    ax_orig.set_title(f'{name}\nUnit square', fontsize=8)
    ax_orig.axhline(0, color='k', lw=0.5)
    ax_orig.axvline(0, color='k', lw=0.5)

    ax_trans.fill(transformed[0], transformed[1], alpha=0.3, color='red')
    ax_trans.plot(transformed[0], transformed[1], 'r-')
    ax_trans.set_xlim(-3.5, 3.5)
    ax_trans.set_ylim(-3.5, 3.5)
    ax_trans.set_aspect('equal')
    ax_trans.set_title(f'Transformed\ndet={det_val:.2f}, area={area:.2f}', fontsize=8)
    ax_trans.axhline(0, color='k', lw=0.5)
    ax_trans.axvline(0, color='k', lw=0.5)

axes[0][0].set_ylabel('Original', fontsize=10)
axes[1][0].set_ylabel('Transformed', fontsize=10)
plt.suptitle('2D: |det(A)| = Area of Parallelogram', fontsize=12)
plt.tight_layout()
plt.show()

# ---- 3D: Unit cube -> parallelepiped ----
print('\n3D Parallelepiped Volumes:')
matrices_3d = [
    ('Identity', np.eye(3)),
    ('Scaling(2,3,4)', np.diag([2., 3., 4.])),
    ('Random', np.array([[1.,2.,0.],[0.,1.,1.],[0.,0.,2.]])),
]

for name, A3 in matrices_3d:
    det_3d = np.linalg.det(A3)
    a1, a2, a3 = A3[:, 0], A3[:, 1], A3[:, 2]
    vol_triple = abs(np.dot(a1, np.cross(a2, a3)))
    print(f'  {name}: |det|={abs(det_3d):.4f}, triple product vol={vol_triple:.4f}')
    assert np.isclose(abs(det_3d), vol_triple, atol=1e-10), \
        f'Volume mismatch for {name}'

In [ ]:
# Verification cell for Exercise 13
# Re-verify area = |det| for all 2D matrices
for name, A2 in matrices_2d:
    unit_square_cols = np.array([[0,1,1,0],[0,0,1,1]], dtype=float)
    det_val = np.linalg.det(A2)
    # Area of parallelogram from columns of A2 = |det(A2)|
    col1 = A2[:, 0]
    col2 = A2[:, 1]
    area_cross = abs(col1[0]*col2[1] - col1[1]*col2[0])  # 2D cross product
    assert np.isclose(abs(det_val), area_cross, atol=1e-10), \
        f'{name}: |det|={abs(det_val):.4f} != area={area_cross:.4f}'

# Re-verify 3D volumes
for name, A3 in matrices_3d:
    det_3d = np.linalg.det(A3)
    a1, a2, a3 = A3[:, 0], A3[:, 1], A3[:, 2]
    vol = abs(np.dot(a1, np.cross(a2, a3)))
    assert np.isclose(abs(det_3d), vol, atol=1e-10), \
        f'{name}: |det|={abs(det_3d):.4f} != triple product={vol:.4f}'

print('Exercise 13 PASSED')
print('  2D: |det(A)| = area of parallelogram verified for all test matrices')
print('  3D: |det(A)| = volume of parallelepiped verified for all test matrices')

## Summary

This lab covered Goodfellow Chapter 2, Sections 2.8–2.11:

**SVD (Section 2.8)**
- Every real matrix admits $A = U \Sigma V^T$ with $U$, $V$ orthogonal and $\Sigma$ diagonal non-negative.
- Geometric interpretation: SVD decomposes a linear map into rotate → scale → rotate.
- Singular values = $\sqrt{\text{eigenvalues of } A^T A}$; right singular vectors = eigenvectors of $A^T A$.
- Truncated SVD gives the best low-rank approximation (Eckart–Young theorem): error = $\sqrt{\sum_{i>k} \sigma_i^2}$.
- Applications: image compression, PCA, dimensionality reduction.

**Moore–Penrose Pseudoinverse (Section 2.9)**
- $A^+ = V \Sigma^+ U^T$: invert non-zero singular values.
- Overdetermined ($m > n$): $A^+ b$ gives least-squares solution minimizing $\|Ax - b\|_2$.
- Underdetermined ($m < n$): $A^+ b$ gives minimum $\|x\|_2$ solution among all solutions.

**Trace (Section 2.10)**
- $\text{tr}(A) = \sum_i A_{ii} = \sum_i \lambda_i$.
- Cyclic property: $\text{tr}(AB) = \text{tr}(BA)$ — enables the **trace trick** used throughout ML.
- $\text{tr}(A^T B) = \langle A, B \rangle_F$ (Frobenius inner product).
- Gradient: $\frac{d}{dA} \text{tr}(AB) = B^T$.

**Determinant (Section 2.11)**
- $\det(A)$: signed volume scaling factor of the linear map $A$.
- Key properties: $\det(AB) = \det(A)\det(B)$, $\det(A^{-1}) = 1/\det(A)$, $\det(cA) = c^n \det(A)$.
- Cofactor expansion: $O(n!)$ — impractical for large matrices.
- LU decomposition: $O(n^3)$ — used in all practical implementations.